# Exp8.0.4 — Phase-aware hierarchical readout with mean-normalized CE

Protocol: `phase_aware_hierarchical_readout_mean_v1`. This notebook is aggregation-only: it reads finalized artifacts and does not train models or refit probes.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

root = Path.cwd()
if root.name == 'notebooks':
    root = root.parent
artifact_dir = root / 'notebooks' / 'artifacts' / 'experiment_8_0_4_phase_aware_hierarchical_readout_mean' / 'phase_aware_hierarchical_readout_mean_v1'
artifact_dir

In [ ]:
method = pd.read_csv(artifact_dir / 'method_summary.csv')
probe = pd.read_csv(artifact_dir / 'probe_summary.csv')
paired = pd.read_csv(artifact_dir / 'paired_delta_summary.csv')
phase = pd.read_csv(artifact_dir / 'phase_structure_summary.csv')
history = pd.read_csv(artifact_dir / 'history_runs.csv')
method[['method', 'native_test_ba_mean', 'native_test_ba_std', 'lif_test_ba_mean', 'lif_penalty_mean']]

## Primary paired comparisons
The first question is whether restoring valid-length mean CE recovers the L2-only baseline. The second is whether `phase_vs_capacity` remains positive.

In [ ]:
paired[['comparison', 'native_test_ba_delta_mean', 'native_test_ba_delta_std', 'lif_test_ba_delta_mean', 'target_probe_test_ba_delta_mean']]

## Representation probes
Show only method-level probe summaries; do not enumerate every seed/run.

In [ ]:
features = ['l1_fixed250', 'l2_fixed250', 'l1_l2_fixed250', 'l2_whole', 'l1fixed250_l2whole']
probe_view = probe[probe.feature.isin(features)].pivot(index='method', columns='feature', values='test_ba_mean')
probe_view

## Phase-bank structure

In [ ]:
phase[['method', 'mean_pairwise_cosine_mean', 'mean_adjacent_cosine_mean', 'between_bin_weight_rms_mean']]

## Training curves — method-level mean only

In [ ]:
curve = history.groupby(['method', 'epoch'], as_index=False)[['train_ba', 'val_ba', 'train_loss', 'val_loss']].mean()
fig, ax = plt.subplots(figsize=(9, 5))
for name, frame in curve.groupby('method'):
    ax.plot(frame.epoch, frame.val_ba, label=name)
ax.set_xlabel('epoch')
ax.set_ylabel('mean validation BA')
ax.legend(fontsize=8)
ax.grid(alpha=0.2)
plt.show()